# wheat-infer-mbv3-d2x2-hflip

Kaggle元データのみで学習したMobileNetV3 Faster R-CNNと2つのEfficientDet-D2。比較元の提出58374318に、epoch別固定検証で重みを選んだD2を追加した。各モデルが元画像と水平反転画像を推論しWBFで統合。モデルの重みは0.25:0.50:0.25。固定検証0.602975（比較0.598586）、独立確認0.573011（比較0.564296）。

In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

spec = json.loads('{"notebook": "yutarotaro/wheat-infer-mbv3-d2x2-hflip", "source_notebook": "submission-mbv3-d2x2-hflip.ipynb", "run_label": "mbv3-d2x2-hflip", "description": "Kaggle元データのみで学習したMobileNetV3 Faster R-CNNと2つのEfficientDet-D2。比較元の提出58374318に、epoch別固定検証で重みを選んだD2を追加した。各モデルが元画像と水平反転画像を推論しWBFで統合。モデルの重みは0.25:0.50:0.25。固定検証0.602975（比較0.598586）、独立確認0.573011（比較0.564296）。", "wheel_dataset": "yutarotaro/wheat-offline-inference-wheels", "wheels": {"effdet-0.4.1-py3-none-any.whl": "10889a226228d515c948e3fcf811e64c0d78d7aa94823a300045653b9c284cb7", "ensemble_boxes-1.0.9-py3-none-any.whl": "f095f34d28034213791b8793b9de4d8ba6e40e6a2a6af02336b84c485d853d6a", "timm-1.0.22-py3-none-any.whl": "888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0"}, "models": [{"trial_id": "8fc98f3e68e8441a-0001", "notebook": "yutarotaro/wheat-train-mbv3-12e-640-hflip-ef473611c538/1", "sha256": "5e2e80f444bd82b2d892eef710836f3fb6cf45d5028ae8bbd97bd062bbbfe1df", "config": {"image_size": 640, "run_label": "mbv3-12e-640-hflip", "epochs": 12, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5}}, {"trial_id": "d31623fdd5114e49-0001", "notebook": "yutarotaro/wheat-train-effdet-d2-6e-768-b995f1851fd0/1", "sha256": "3d565f728235bb868541a55e6bae1248dc2a71167d480dbce3dadf3f83dc5bf3", "config": {"detector": "efficientdet-d2", "image_size": 768, "run_label": "effdet-d2-6e-768", "epochs": 6, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "mosaic": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5}}, {"trial_id": "1d692cda0f6e4cd2-0001", "notebook": "yutarotaro/wheat-train-effdet-d2-6e-768-epoch-sc-504712498bd0/1", "sha256": "f95951573cca7fb7a6a048e7b78165284f7cf058b531d42c21425ac5503ae099", "config": {"detector": "efficientdet-d2", "image_size": 768, "run_label": "effdet-d2-6e-768-epoch-score", "epochs": 6, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "mosaic": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5, "use_training_supplement": false}}], "weights": [0.25, 0.5, 0.25], "confidence": 0.34, "iou": 0.45, "validation_score": 0.6029749020090281, "holdout_score": 0.5730111940999111, "tta_flips": ["horizontal"], "tta_iou": 0.55, "validation_by_source": {"rres_1": 0.7114418296500798, "arvalis_2": 0.37328023171033065}, "holdout_by_source": {"inrae_1": 0.6483249793746219, "arvalis_3": 0.5492988037450828}, "variant": "add-detector", "primary_trial_id": "1d692cda0f6e4cd2-0001"}')
Path("submission-spec.json").write_text(json.dumps(spec))
wheel_paths = []
for name, expected in spec["wheels"].items():
    (wheel,) = Path("/kaggle/input").rglob(name)
    with wheel.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
            raise ValueError(f"Wheel checksum mismatch: {name}")
    wheel_paths.append(str(wheel))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                "--force-reinstall", *wheel_paths], check=True)


## data.py

In [ ]:
%%writefile data.py
"""Validate annotations and freeze a split by acquisition source."""

import argparse
import csv
import hashlib
import itertools
import json
import math
import random
from collections import Counter
from pathlib import Path

from PIL import Image


def sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def load_dataset(root):
    images = {}
    with (root / "train.csv").open(newline="") as handle:
        for row in csv.DictReader(handle):
            image_id, source = row["image_id"], row["source"]
            if not image_id or not source or Path(image_id).name != image_id:
                raise ValueError(f"Invalid image id or source: {image_id!r}, {source!r}")
            width, height = int(row["width"]), int(row["height"])
            if min(width, height) <= 0:
                raise ValueError(f"Invalid image dimensions: {image_id}")
            x, y, w, h = map(float, json.loads(row["bbox"]))
            if not all(map(math.isfinite, [x, y, w, h])) or min(x, y) < 0 or min(w, h) <= 0:
                raise ValueError(f"Invalid bounding box: {image_id}")
            if x + w > width + 1e-6 or y + h > height + 1e-6:
                raise ValueError(f"Out-of-bounds annotation: {image_id}")
            record = images.setdefault(image_id, dict(source=source, width=width, height=height, boxes=[]))
            if (record["source"], record["width"], record["height"]) != (source, width, height):
                raise ValueError(f"Inconsistent metadata: {image_id}")
            record["boxes"].append([x, y, min(x + w, width), min(y + h, height)])

    paths = {p.stem: p for p in (root / "train").glob("*.jpg")}
    missing = images.keys() - paths.keys()
    if missing:
        raise ValueError(f"Missing annotated images: {sorted(missing)[:5]}")
    unlabelled = sorted(paths.keys() - images.keys())
    if unlabelled:
        print(f"Excluding {len(unlabelled)} images with no source/annotation in train.csv.")
    checksums = {}
    for image_id, record in sorted(images.items()):
        path = paths[image_id]
        with Image.open(path) as im:
            if im.size != (record["width"], record["height"]):
                raise ValueError(f"Unexpected image dimensions: {image_id}")
        checksums[image_id] = sha(path)
    return images, checksums


def split_dataset(images, checksums, seed=42):
    parent = {r["source"]: r["source"] for r in images.values()}

    def group(s):
        while s != parent[s]:
            s = parent[s]
        return s

    seen = {}
    for image_id in sorted(images):
        source = images[image_id]["source"]
        previous = seen.setdefault(checksums[image_id], source)
        parent[group(source)] = group(previous)

    sizes = Counter(group(row["source"]) for row in images.values())
    groups = sorted(sizes)
    if len(groups) < 3:
        raise ValueError("Need at least three independent source groups.")
    rng = random.Random(seed)
    rng.shuffle(groups)
    target = (0.6, 0.2, 0.2)
    if len(groups) <= 10:
        assignments = itertools.product(range(3), repeat=len(groups))
    else:
        assignments = (rng.choices(range(3), target, k=len(groups)) for _ in range(10_000))
    winner, error = None, math.inf
    for assignment in assignments:
        if set(assignment) != {0, 1, 2}:
            continue
        counts = [sum(sizes[g] for g, part in zip(groups, assignment) if part == i) for i in range(3)]
        if counts[0] < max(counts[1:]):
            continue
        loss = sum((n / len(images) - goal) ** 2 for n, goal in zip(counts, target))
        if loss < error:
            winner, error = assignment, loss
    if winner is None:
        raise ValueError("No valid source-group split.")
    labels = {
        g: ("train", "validation", "holdout")[part] for g, part in zip(groups, winner)}
    partitions = {part: [] for part in ("train", "validation", "holdout")}
    for image_id, row in sorted(images.items()):
        partitions[labels[group(row["source"])]].append(image_id)
    return partitions


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, default=Path("data/raw"))
    parser.add_argument("--output", type=Path, default=Path("data/prepared/split.json"))
    args = parser.parse_args()
    if args.output.exists():
        parser.error("The split is already frozen; use a different output path to create a new evaluation cohort.")
    images, hashes = load_dataset(args.data)
    split = split_dataset(images, hashes)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    payload = {"seed": 42, "csv_sha256": sha(args.data / "train.csv"), "images": hashes, "partitions": split}
    with args.output.open("x") as handle:
        json.dump(payload, handle, sort_keys=True, indent=2)
    for part, ids in split.items():
        print(part, len(ids), dict(Counter(images[i]["source"] for i in ids)))


if __name__ == "__main__":
    main()


## metric.py

In [ ]:
%%writefile metric.py
"""Image-averaged wheat precision over IoU thresholds 0.50–0.75."""

import math

THRESHOLDS = tuple(n / 100 for n in range(50, 76, 5))


def area(box):
    return (box[2] - box[0]) * (box[3] - box[1])


def iou(a, b):
    overlap = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return overlap / (area(a) + area(b) - overlap)


def validate_predictions(rows, width, height):
    if not isinstance(rows, list):
        raise ValueError("Predictions must be a list.")
    for row in rows:
        if not isinstance(row, list) or len(row) != 5 or not all(
            isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) for x in row
        ):
            raise ValueError(f"Expected finite [confidence,x1,y1,x2,y2]: {row!r}")
        confidence, x1, y1, x2, y2 = row
        if not (0 <= confidence <= 1 and 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError(f"Invalid prediction: {row!r}")


def match(actual, predicted, threshold):
    remaining = set(range(len(actual)))
    hits, extras, duplicates = [], [], []
    for j in sorted(range(len(predicted)), key=lambda n: -predicted[n][0]):
        box = predicted[j][1:]
        overlaps = [iou(box, truth) for truth in actual]
        i = max(remaining, key=lambda k: (overlaps[k], -k), default=None)
        if i is not None and overlaps[i] > threshold:
            remaining.remove(i)
            hits.append(j)
        elif max(overlaps, default=0) > threshold:
            duplicates.append(j)
        else:
            extras.append(j)
    return hits, extras, duplicates, sorted(remaining)


def image_score(actual, predicted):
    if not actual and not predicted:
        return None
    scores = []
    for threshold in THRESHOLDS:
        hits, extra, duplicates, misses = match(actual, predicted, threshold)
        scores.append(len(hits) / (len(hits) + len(extra) + len(duplicates) + len(misses)))
    return sum(scores) / len(scores)


## fusion.py

In [ ]:
%%writefile fusion.py
"""Weighted Boxes Fusion for validated wheat detections in image coordinates."""

from ensemble_boxes.ensemble_boxes_wbf import weighted_boxes_fusion

from metric import validate_predictions


def fuse(rows, width, height, weights, threshold):
    scale = (width, height, width, height)
    for guesses in rows:
        validate_predictions(guesses, width, height)
    # Zero-confidence boxes carry no weight when averaging positions.
    rows = [[box for box in guesses if box[0] > 0] for guesses in rows]
    boxes, scores, _ = weighted_boxes_fusion(
        [[[value / size for value, size in zip(box[1:], scale)] for box in guesses] for guesses in rows],
        [[box[0] for box in guesses] for guesses in rows],
        [[1] * len(guesses) for guesses in rows],
        weights=weights, iou_thr=threshold, conf_type="box_and_model_avg",
    )
    # The library uses float32 cluster averages, which can round above 1.
    return [[min(float(conf), 1.0), *(min(float(value), 1.0) * size for value, size in zip(box, scale))]
            for box, conf in zip(boxes, scores)]


## model.py

In [ ]:
%%writefile model.py
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

import torch
from torch import nn
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_convert
from torchvision.transforms.functional import normalize, resize

EFFDET_PACKAGES = {"effdet": "0.4.1", "timm": "1.0.22"}
RTDETR_PACKAGES = {"scipy": "1.16.3", "transformers": "4.57.6"}


def prepare_packages(packages, group):
    missing = []
    for name, expected in packages.items():
        try:
            installed = version(name)
        except PackageNotFoundError:
            installed = None
        if installed != expected:
            missing.append(f"{name}=={expected}")
    if missing:
        if os.environ.get("ML_LOOP_REMOTE") != "kaggle":
            raise RuntimeError(f"Install the {group} group with uv sync --group wheat --group {group}.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        *missing], check=True)


class EfficientWheat(nn.Module):
    def __init__(self, config, pretrained):
        super().__init__()
        prepare_packages(EFFDET_PACKAGES, "wheat")
        from effdet import DetBenchPredict, DetBenchTrain, create_model

        self.size = config["image_size"]
        self.confidence = config["confidence"]
        if config["nms"] != 0.5:
            raise ValueError("EfficientDet uses NMS at IoU 0.50.")
        if self.size % 128:
            raise ValueError("EfficientDet image size must be a multiple of 128.")
        network = create_model("tf_efficientdet_d2", num_classes=1, pretrained=pretrained,
                               pretrained_backbone=False, image_size=(self.size, self.size),
                               max_det_per_image=300)
        self.loss_bench = DetBenchTrain(network)
        self.predict_bench = DetBenchPredict(network)

    def forward(self, images, labels=None):
        shapes = [image.shape[-2:] for image in images]
        inputs = torch.stack([
            normalize(resize(image, [self.size, self.size], antialias=True),
                      [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            for image in images
        ])
        if labels is not None:
            boxes = []
            for label, (h, w) in zip(labels, shapes, strict=True):
                gt = label["boxes"][:, [1, 0, 3, 2]]
                boxes.append(gt * gt.new_tensor([self.size / h, self.size / w] * 2))
            info = {
                "bbox": boxes, "cls": [label["labels"] for label in labels],
                "img_scale": inputs.new_ones(len(images)),
                "img_size": inputs.new_full((len(images), 2), self.size),
            }
            return {"loss": self.loss_bench(inputs, info)["loss"]}

        detections = self.predict_bench(inputs)
        outputs = []
        for prediction, (h, w) in zip(detections, shapes, strict=True):
            if not torch.isfinite(prediction).all():
                raise ValueError("Non-finite EfficientDet prediction.")
            scores = prediction[:, 4]
            boxes = prediction[:, :4].clamp(0, self.size)
            boxes = boxes * boxes.new_tensor([w / self.size, h / self.size] * 2)
            keep = (scores >= self.confidence) & (boxes[:, 2:] > boxes[:, :2]).all(dim=1)
            outputs.append({"boxes": boxes[keep], "scores": scores[keep]})
        return outputs


class RTDetrWheat(nn.Module):
    def __init__(self, config, pretrained):
        super().__init__()
        prepare_packages(RTDETR_PACKAGES, "detr")
        from transformers import RTDetrV2Config, RTDetrV2ForObjectDetection

        self.size = config["image_size"]
        self.confidence = config["confidence"]
        if self.size % 32:
            raise ValueError("RT-DETR requires a multiple-of-32 input size.")
        conditions = RTDetrV2Config.from_dict(config["detr_architecture"])
        conditions.num_labels = 1
        conditions.id2label = {0: "wheat_head"}
        conditions.label2id = {"wheat_head": 0}
        if pretrained:
            self.detector = RTDetrV2ForObjectDetection.from_pretrained(
                config["pretrained_model"], revision=config["pretrained_revision"], config=conditions,
                use_safetensors=True, ignore_mismatched_sizes=True,
            )
        else:
            self.detector = RTDetrV2ForObjectDetection(conditions)

    def parameter_groups(self, backbone_lr):
        backbone = list(self.detector.model.backbone.parameters())
        ids = {id(param) for param in backbone}
        return [{"params": backbone, "lr": backbone_lr},
                {"params": [param for param in self.parameters() if id(param) not in ids]}]

    def forward(self, images, labels=None):
        shapes = [image.shape[-2:] for image in images]
        pixels = torch.stack([resize(image, [self.size, self.size], antialias=True) for image in images])
        targets = None
        if labels is not None:
            targets = []
            for label, (height, width) in zip(labels, shapes, strict=True):
                boxes = label["boxes"]
                scaled = boxes / boxes.new_tensor([width, height, width, height])
                targets.append({"boxes": box_convert(scaled, "xyxy", "cxcywh"),
                                "class_labels": label["labels"] - 1})
        outputs = self.detector(pixel_values=pixels, labels=targets)
        if labels is not None:
            return {"loss": outputs.loss}
        detections = []
        for scores, boxes, (height, width) in zip(
            outputs.logits[..., 0].sigmoid(), outputs.pred_boxes, shapes, strict=True
        ):
            boxes = box_convert(boxes, "cxcywh", "xyxy").clamp(0, 1)
            boxes = boxes * boxes.new_tensor([width, height, width, height])
            if not torch.isfinite(boxes).all() or not torch.isfinite(scores).all():
                raise ValueError("Non-finite RT-DETR detection.")
            keep = (scores >= self.confidence) & (boxes[:, 2:] > boxes[:, :2]).all(dim=1)
            detections.append({"boxes": boxes[keep], "scores": scores[keep]})
        return detections


def create_model(config, pretrained=True):
    detector = config.get("detector", "fasterrcnn-mobilenet-v3")
    if detector == "efficientdet-d2":
        return EfficientWheat(config, pretrained)
    if detector == "rtdetr-v2-r18":
        return RTDetrWheat(config, pretrained)
    if detector == "fasterrcnn-mobilenet-v3":
        build, weights = fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
    elif detector == "fasterrcnn-resnet50-v2":
        build, weights = fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    else:
        raise ValueError(f"Unknown detector: {detector}")
    model = build(
        weights=weights if pretrained else None, weights_backbone=None, min_size=config["image_size"],
        max_size=config["image_size"], box_score_thresh=config["confidence"],
        box_nms_thresh=config["nms"], box_detections_per_img=300,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, 2)
    return model


## proposals.py

In [ ]:
%%writefile proposals.py
"""Full-image and overlapping tile proposals for wheat inference."""

import math

import numpy as np

from fusion import fuse
from metric import validate_predictions


def tile_predictions(model, image, spec, batch_size=4):
    height, width = image.shape[-2:]
    window = spec["window_fraction"]
    margin = spec["seam_margin"]
    if not 0.5 < window < 1 or not 0 <= margin < window - 0.5:
        raise ValueError("Expected overlapping tiles with a smaller seam margin.")
    tw, th = math.ceil(window * width), math.ceil(window * height)
    origins = [(x, y) for y in (0, height - th) for x in (0, width - tw)]
    rows = []
    for offset in range(0, len(origins), batch_size):
        positions = origins[offset:offset + batch_size]
        detections = model([image[:, y:y + th, x:x + tw] for x, y in positions])
        for (x, y), result in zip(positions, detections, strict=True):
            for score, (left, top, right, bottom) in zip(
                result["scores"].tolist(), result["boxes"].tolist(), strict=True
            ):
                if ((x > 0 and left < margin * width) or
                    (y > 0 and top < margin * height) or
                    (x + tw < width and right > tw - margin * width) or
                    (y + th < height and bottom > th - margin * height)):
                    continue
                if (right - left) * (bottom - top) <= spec["max_area"] * width * height:
                    rows.append([score, x + left, y + top, x + right, y + bottom])
    return fuse([rows], width, height, [1], spec["iou"])


def overlaps(boxes, other):
    lo = np.maximum(boxes[:, None, :2], other[None, :, :2])
    hi = np.minimum(boxes[:, None, 2:], other[None, :, 2:])
    joint = (hi - lo).clip(min=0).prod(axis=2)
    area = (boxes[:, 2:] - boxes[:, :2]).prod(axis=1)
    other_area = (other[:, 2:] - other[:, :2]).prod(axis=1)
    return joint / (area[:, None] + other_area[None, :] - joint)


def as_array(rows):
    return np.asarray(rows, dtype=np.float64).reshape(-1, 5)


def tile_rescue(baseline, tiles, spec):
    if not tiles:
        return baseline
    guesses = as_array(tiles)
    present = (overlaps(guesses[:, 1:], as_array(baseline)[:, 1:]).max(axis=1)
               if baseline else np.zeros(len(tiles)))
    return [*baseline, *guesses[(guesses[:, 0] >= spec["rescue_confidence"]) &
                               (present < spec["rescue_iou"])].tolist()]


def quality_features(candidates, sources, width, height):
    boxes = candidates[:, 1:]
    coords = boxes / [width, height, width, height]
    span = coords[:, 2:] - coords[:, :2]
    border = np.minimum(coords[:, :2], 1 - coords[:, 2:]).min(axis=1)
    columns = [candidates[:, 0], np.log(span[:, 0]), np.log(span[:, 1]), border]
    for rows in sources:
        raw = as_array(rows)
        support = np.zeros(len(candidates))
        agreement = np.zeros(len(candidates))
        if len(raw):
            similarity = overlaps(boxes, raw[:, 1:])
            best = similarity.argmax(axis=1)
            agreement = similarity[np.arange(len(best)), best]
            support = np.where(agreement >= 0.25, raw[best, 0], 0)
        columns.extend((agreement, support, agreement * support,
                        np.full(len(candidates), np.log1p(len(raw)))))
    return np.column_stack(columns)


def quality_candidates(global_rows, tiles, width, height, weights, threshold, with_tiles):
    sources = [*global_rows, tiles] if with_tiles else global_rows
    global_fused = fuse(global_rows, width, height, weights, threshold)
    # Keep alternative coordinates until the predicted quality determines NMS order.
    candidates = as_array([*global_fused, *(box for rows in sources for box in rows)])
    return candidates, quality_features(candidates, sources, width, height)


def rescore(candidates, features, width, height, spec):
    if not len(candidates):
        return []
    scores = np.clip(spec["intercept"] + features @ spec["coefficients"], 0, 1)
    remaining = np.argsort(-scores, kind="stable")
    remaining = remaining[scores[remaining] >= spec["confidence"]]
    selected = []
    while len(remaining):
        first, remaining = remaining[0], remaining[1:]
        selected.append(first)
        similarity = overlaps(candidates[[first], 1:], candidates[remaining, 1:])[0]
        remaining = remaining[similarity < spec["nms_iou"]]
    rows = np.column_stack((scores[selected], candidates[selected, 1:])).tolist()
    validate_predictions(rows, width, height)
    return rows


def refine_with_tiles(baseline, candidates, features, tiles, spec):
    """Use learned box quality to refine supported detections and rescue novel ones."""
    if not tiles:
        return baseline
    tile_count = len(tiles)
    tile_boxes = as_array(tiles)[:, 1:]
    quality = np.clip(spec["intercept"] + features @ spec["coefficients"], 0, 1)
    tile_quality = quality[-tile_count:]
    reference = as_array(baseline)
    similarities = overlaps(reference[:, 1:], tile_boxes)
    updated = [row.copy() for row in baseline]
    if len(reference):
        owner = similarities.argmax(axis=0)
        nearest = similarities.max(axis=0)
        current = quality[:len(reference)]
        # The baseline is the confidence-filtered prefix of the sorted WBF candidates.
        if not np.allclose(reference, candidates[:len(reference)]):
            raise ValueError("Expected baseline detections to prefix the quality candidates.")
        for index, row in enumerate(reference):
            supported = (owner == index) & (similarities[index] >= 0.50)
            eligible = np.flatnonzero(supported & (tile_quality >= current[index] + 0.05))
            if len(eligible):
                best = eligible[tile_quality[eligible].argmax()]
                updated[index][1:] = ((row[1:] + tile_boxes[best]) / 2).tolist()
    else:
        nearest = np.zeros(tile_count)
    for index in np.flatnonzero((nearest < 0.30) & (tile_quality >= spec["confidence"])):
        updated.append([float(tile_quality[index]), *tile_boxes[index].tolist()])
    return updated


## submission.py

In [ ]:
%%writefile submission.py
"""Write Kaggle predictions from frozen detector checkpoints."""

import csv
import json
import math
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

from data import sha
from fusion import fuse
from metric import validate_predictions
from model import create_model
from proposals import quality_candidates, refine_with_tiles, rescore, tile_predictions, tile_rescue


def kaggle_inputs(mounted, spec):
    roots = sorted(path.parent for path in mounted.rglob("sample_submission.csv")
                   if path.parent.name == "global-wheat-detection")
    if len(roots) != 1:
        raise ValueError("Expected one mounted competition dataset.")
    by_hash = {sha(path): path for path in mounted.rglob("weights.pt")}
    return roots[0], [by_hash[model["sha256"]] for model in spec["models"]]


def prediction_string(predictions, width, height):
    validate_predictions(predictions, width, height)
    tokens = []
    for score, x1, y1, x2, y2 in predictions:
        left, top = math.floor(x1), math.floor(y1)
        right, bottom = math.ceil(x2), math.ceil(y2)
        tokens.append(f"{score:.8f} {left} {top} {right-left} {bottom-top}")
    return " ".join(tokens)


def predict_views(model, tensor, flips, iou):
    height, width = tensor.shape[-2:]
    predictions = []
    for view in ("original", *flips):
        if view == "original":
            result = model([tensor])[0]
            boxes = result["boxes"]
        else:
            dimension, extent = {"horizontal": (-1, width), "vertical": (-2, height)}[view]
            result = model([tensor.flip(dims=(dimension,))])[0]
            boxes = result["boxes"].clone()
            columns = (0, 2) if view == "horizontal" else (1, 3)
            boxes[:, columns] = extent - boxes[:, columns[::-1]]
        predictions.append([[score, *box] for score, box in
                            zip(result["scores"].tolist(), boxes.tolist(), strict=True)])
    return fuse(predictions, width, height, [1] * len(predictions), iou) if flips else predictions[0]


@torch.inference_mode()
def predict(data_root, checkpoints, spec, output, device):
    if len(checkpoints) != len(spec["models"]):
        raise ValueError("Expected one checkpoint per model.")
    models = []
    for path, conditions in zip(checkpoints, spec["models"], strict=True):
        if sha(path) != conditions["sha256"]:
            raise ValueError(f"Checkpoint mismatch for {conditions['trial_id']}.")
        model = create_model(conditions["config"], pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        models.append(model.eval())
    paths = {path.stem: path for path in (data_root / "test").glob("*.jpg")}
    views = [conditions.get("tta_flips", spec.get("tta_flips", [])) for conditions in spec["models"]]
    for flips in views:
        if len(set(flips)) != len(flips) or set(flips) - {"horizontal", "vertical"}:
            raise ValueError("Expected a unique list of horizontal or vertical TTA flips.")
    with (data_root / "sample_submission.csv").open(newline="") as handle:
        ids = [row["image_id"] for row in csv.DictReader(handle)]
    if not ids or len(ids) != len(paths) or set(ids) != paths.keys():
        raise ValueError("Sample submission and test images must contain the same unique IDs.")
    temporary = output.with_suffix(".tmp")
    with temporary.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(("image_id", "PredictionString"))
        for key in ids:
            with Image.open(paths[key]) as image:
                width, height = image.size
                tensor = pil_to_tensor(image.convert("RGB")).float().div_(255).to(device)
            rows = [predict_views(model, tensor, flips, spec.get("tta_iou", 0.55))
                    for model, flips in zip(models, views, strict=True)]
            tiles = spec.get("tiles")
            tile_rows = tile_predictions(models[tiles["model_index"]], tensor, tiles) if tiles else []
            quality = spec.get("quality")
            if len(rows) == 1:
                detections = rows[0]
            else:
                detections = fuse(rows, width, height, spec["weights"], spec["iou"])
            detections = [box for box in detections if box[0] >= spec["confidence"]]
            if quality:
                candidates, features = quality_candidates(rows, tile_rows, width, height,
                                                         spec["weights"], spec["iou"], bool(tiles))
                if quality.get("refine_with_tiles"):
                    detections = refine_with_tiles(detections, candidates, features, tile_rows, quality)
                else:
                    detections = rescore(candidates, features, width, height, quality)
            elif tiles:
                detections = tile_rescue(detections, tile_rows, tiles)
            writer.writerow((key, prediction_string(detections, width, height)))
    temporary.replace(output)
    print(f"Validated and wrote {len(ids)} predictions")


def run_kaggle(spec_path):
    spec = json.loads(spec_path.read_text())
    data, checkpoints = kaggle_inputs(Path("/kaggle/input"), spec)
    torch.set_num_threads(2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    predict(data, checkpoints, spec, Path("/kaggle/working/submission.csv"), device)


## 推論と提出用CSVの作成

In [ ]:
from submission import run_kaggle

run_kaggle(Path("submission-spec.json"))
